In [196]:
%load_ext autoreload
%autoreload 2

# Objective

The purpose of this notebook is to demonstrate kernel products and addition using the HRR algebra.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sspspace

from matplotlib import cm
c_map = cm.jet

import sys
sys.path.append('../experiments/')

# import figure_utils



In [131]:
def make_laplace_freqs(dim, eps=1e-3, rng=np.random):
    a = rng.standard_cauchy(size=(dim - 1) // 2)
    #a = rng.exponential(scale=(dim-1) // 2)
    sign = rng.choice((-1, +1), len(a))
    phi = sign * np.pi * (eps + a * (1 - 2 * eps))

    fv = np.zeros(dim, dtype='complex64')
    
    fv[0] = 1
    fv[1:(dim + 1) // 2] = phi #np.cos(phi) + 1j * np.sin(phi)
    fv[-1:dim // 2:-1] = -phi #np.conj(fv[1:(dim + 1) // 2])
    if dim % 2 == 0:
        fv[dim // 2] = 1
    return fv, phi


laplace_phase_mat, a = make_laplace_freqs(1000)
laplace_enc = sspspace.SSPEncoder(laplace_phase_mat[:,None], length_scale=2)

origin = laplace_enc.encode([[0]])
xs = np.linspace(-4,4,1000)

phis = laplace_enc.encode(xs[:,None])

sims = phis | origin

# plt.subplot(1,3,1)
# plt.hist(a, bins=100, density=True)
# plt.title('Frequency histograms')
# plt.subplot(1,3,2)
plt.plot(xs, sims)
plt.title('Similarity')
# plt.subplot(1,3,3)
# ffts = np.fft.fft(phis, axis=1)
# mean_mags = np.mean(np.abs(ffts), axis=0)
# std_mags = np.std(np.abs(ffts), axis=0)
# plt.plot(mean_mags)
# plt.fill_between(np.arange(1000), y1=mean_mags - std_mags, y2=mean_mags + std_mags, alpha = 0.2)
# plt.title('Fourier coefficient\n magnitude')


In [3]:
def make_sin_freqs(dim, eps=1e-3, rng=np.random):
#     a = rng.standard_cauchy(size=(dim - 1) // 2)
    a = np.ones(((dim-1) // 2,)) #* np.pi
    sign = rng.choice((-1, +1), len(a))
    phi = sign * np.pi * (eps + a * (1 - 2 * eps))

    fv = np.zeros(dim, dtype='complex64')
    
    fv[0] = 0
    fv[1:(dim + 1) // 2] = phi #np.cos(phi) + 1j * np.sin(phi)
    fv[-1:dim // 2:-1] = -phi #np.conj(fv[1:(dim + 1) // 2])
    if dim % 2 == 0:
        fv[dim // 2] = 0
    return fv, phi

sin_phase_mat, a = make_sin_freqs(1000)
sin_enc = sspspace.SSPEncoder(sin_phase_mat[:,None], length_scale=1)

origin = sin_enc.encode([[0]])
xs = np.linspace(-4,4,1000)

phis = sin_enc.encode(xs[:,None])

sims = phis | origin

plt.subplot(1,3,1)
plt.hist(a, bins=100, density=True)
plt.title('Frequency histograms')
plt.subplot(1,3,2)
plt.plot(xs, sims)
plt.title('Similarity')
plt.subplot(1,3,3)
ffts = np.fft.fft(phis, axis=1)
mean_mags = np.mean(np.abs(ffts), axis=0)
std_mags = np.std(np.abs(ffts), axis=0)
plt.plot(mean_mags)
plt.fill_between(np.arange(1000), y1=mean_mags - std_mags, y2=mean_mags + std_mags, alpha = 0.2)
plt.title('Fourier coefficient\n magnitude')


In [103]:
def linear_kernel(x,x_p, l=1,sigma_v=1, sigma_b=1):
    return sigma_b + sigma_v*(x-l)*(x_p-l)


plt.plot(np.linspace(-2,2,100), linear_kernel(0,np.linspace(-2,2,100),l=1))

ks = linear_kernel(query_xs[:,0],query_xs[:,1], l=-6)
# plt.imshow(ks.reshape((100,100)), origin='lower')
# plt.colorbar()

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.plot_surface(X1, X2, ks.reshape((100,100)), cmap=c_map)

In [113]:
from scipy.interpolate import interp1d

def get_lin_kernel_psd(num_samples=1000):
    xs = np.linspace(-1,1,1000)
    ys = np.copy(xs)
#     ys = linear_kernel(0,xs)
    
    fft = np.fft.fft(ys)
    fft_freqs = np.fft.fftfreq(ys.size,d=((xs.max()-xs.min())/xs.size))
    
    fft = np.fft.fftshift(fft)
    fft_freqs = np.fft.fftshift(fft_freqs)
    
    psd = np.abs(fft)**2 / np.sum(np.abs(fft)**2)    
    psd_func = interp1d(fft_freqs, psd)

    plt.plot(fft_freqs, psd)    
    plt.show()
#     print(psd.max())

    ret_freqs = np.random.choice(fft_freqs, size=(num_samples,), p=psd)
    ret_probs = psd_func(ret_freqs)
    return ret_freqs, ret_probs
    
freqs, probs = get_lin_kernel_psd()
    
plt.subplot(1,2,1)
plt.hist(freqs,bins=500, density=True)

plt.subplot(1,2,2)
plt.plot(np.linspace(-2,2,100), linear_kernel(0,np.linspace(-2,2,100),l=-1))
    

In [115]:
def make_lin_freqs(dim, eps=1e-3, rng=np.random):
    a, a_probs = get_lin_kernel_psd(num_samples=(dim-1) // 2)
    sign = rng.choice((-1, +1), len(a))
    phi = sign * np.pi * (eps + a * (1 - 2 * eps))

    fv = np.zeros(dim, dtype='complex64')
    
    dc_val = 1#0.9216446858029479 
    fv[0] = dc_val
    fv[1:(dim + 1) // 2] = phi
    fv[-1:dim // 2:-1] = -phi
    if dim % 2 == 0:
        fv[dim // 2] = dc_val
    return fv, phi

lin_phase_mat, a = make_lin_freqs(2000)
lin_enc = sspspace.SSPEncoder(lin_phase_mat[:,None], length_scale=1)

origin = lin_enc.encode([[0]])
xs = np.linspace(-4,4,1000)

phis = lin_enc.encode(xs[:,None])

sims = phis | origin

plt.subplot(1,3,1)
plt.hist(a, bins=100, density=True)
plt.title('Frequency histograms')
plt.subplot(1,3,2)
plt.plot(xs, sims)
plt.title('Similarity')
plt.subplot(1,3,3)
ffts = np.fft.fft(phis, axis=1)
mean_mags = np.mean(np.abs(ffts), axis=0)
std_mags = np.std(np.abs(ffts), axis=0)
plt.plot(mean_mags)
plt.fill_between(np.arange(ffts.shape[1]), y1=mean_mags - std_mags, y2=mean_mags + std_mags, alpha = 0.2)
plt.title('Fourier coefficient\n magnitude')

In [166]:
compositional_fig = plt.figure(figsize=(10, 9))
subfigs = compositional_fig.subfigures(2, 1, wspace=0.07, height_ratios=[1, 2])
font_size = 'xx-large'

In [167]:
laplace_origin = laplace_enc.encode([[0]])
sin_origin = sin_enc.encode([[0]])
origin = laplace_origin * sin_origin


xs = np.linspace(-4,4,200)

laplace_phis = laplace_enc.encode(xs[:,None])
sin_phis = sin_enc.encode(xs[:,None])

phis = laplace_phis * sin_phis

sims = phis | origin
laplace_sims = laplace_phis | laplace_origin
sin_sims = sin_phis | sin_origin

sum_phis = (laplace_phis + sin_phis)
sum_origin = (laplace_origin + sin_origin)
sum_sims = sum_phis | sum_origin



axs_top = subfigs[0].subplots(1, 4)
# plt.figure(figsize=(10,3))
# plt.subplot(1,4,1)
axs_top[0].plot(xs, sin_sims)
axs_top[0].spines['top'].set_visible(False)
axs_top[0].spines['right'].set_visible(False)
axs_top[0].set_xticklabels([])
axs_top[0].set_yticks([])
axs_top[0].set_title('PER', fontsize=font_size)

# plt.subplot(1,4,2)
axs_top[1].plot(xs, laplace_sims, lw=2)
axs_top[1].spines['top'].set_visible(False)
axs_top[1].spines['right'].set_visible(False)
axs_top[1].set_xticklabels([])
axs_top[1].set_yticks([])
axs_top[1].set_title('LaPlace', fontsize=font_size)

# plt.subplot(1,4,3)
axs_top[2].plot(xs, sims, lw=2)
axs_top[2].spines['top'].set_visible(False)
axs_top[2].spines['right'].set_visible(False)
axs_top[2].set_title(r'PER $\circledast$ LaPlace', fontsize=font_size)
axs_top[2].set_xticklabels([])
axs_top[2].set_yticks([])


# plt.subplot(1,4,4)
axs_top[3].plot(xs, sum_sims, lw=2)
axs_top[3].spines['top'].set_visible(False)
axs_top[3].spines['right'].set_visible(False)
axs_top[3].set_title(r'PER + LaPlace', fontsize=font_size)
axs_top[3].set_xticklabels([])
axs_top[3].set_yticks([])

# plt.show()

# kernels_1d = plt.gcf()

# figure_utils.save(kernels_1d, 'prod-and-sum-1d.pdf')


# Extending to 2D

Want to show some fancier figures.

In [6]:
length_scale = 4

x1_space = sspspace.RandomSSPSpace(domain_dim=1,ssp_dim=1000)
x1_space.update_lengthscale([length_scale])
x2_space = sspspace.RandomSSPSpace(domain_dim=1,ssp_dim=1000)
x2_space.update_lengthscale([length_scale])

x1s = np.linspace(-20,20,100)
x2s = np.linspace(-20,20,100)
X1, X2 = np.meshgrid(x1s, x2s)
query_xs = np.vstack((X1.flatten(), X2.flatten())).T

x1_origin = x1_space.encode([[0]])
x2_origin = x2_space.encode([[0]])

prod_origin = x1_origin * x2_origin
sum_origin = (x1_origin + x2_origin ) / 2

query1_phis = x1_space.encode(query_xs[:,0][:,None])
query2_phis = x2_space.encode(query_xs[:,1][:,None])

query_sum_phis = (query1_phis + query2_phis) / 2
query_prod_phis = query1_phis * query2_phis

sum_sims = query_sum_phis | sum_origin
prod_sims = query_prod_phis | prod_origin

In [125]:

# plt.style.use('_mpl-gallery')

# Plot the surface
fig, ax = plt.subplots(ncols=2, subplot_kw={"projection": "3d"})
ax[0].plot_surface(X1, X2, sum_sims.reshape((100,100)), cmap=c_map)
# ax.plot_surface(X, Y, Z, vmin=Z.min() * 2, cmap=cm.Blues)

ax[0].set(xticklabels=[],
       yticklabels=[],
       zticklabels=[])
ax[0].set_title(r'$\mathrm{sinc}(x_{1}) + \mathrm{sinc}(x_{2})$')
ax[0].set_xlabel(r'$x_{1}$')
ax[0].set_ylabel(r'$x_{2}$')
ax[0].grid(False)
ax[0].xaxis.labelpad = -10
ax[0].yaxis.labelpad = -10


ax[1].plot_surface(X1, X2, prod_sims.reshape((100,100)), cmap=c_map)
# ax.plot_surface(X, Y, Z, vmin=Z.min() * 2, cmap=cm.Blues)

ax[1].set(xticklabels=[],
       yticklabels=[],
       zticklabels=[])
ax[1].set_title(r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$')
ax[1].set_xlabel(r'$x_{1}$')
ax[1].set_ylabel(r'$x_{2}$')
ax[1].grid(False)
ax[1].xaxis.labelpad = -10
ax[1].yaxis.labelpad = -10



plt.show()

In [168]:
# Plot the surface
# fig, ax = plt.subplots(nrows=2, ncols=3, subplot_kw={"projection": "3d"})
length_scale = 4

x1_space = sspspace.RandomSSPSpace(domain_dim=1,ssp_dim=1000)
x1_space.update_lengthscale([length_scale])
x2_space = sspspace.RandomSSPSpace(domain_dim=1,ssp_dim=1000)
x2_space.update_lengthscale([length_scale])

x1s = np.linspace(-20,20,100)
x2s = np.linspace(-20,20,100)
X1, X2 = np.meshgrid(x1s, x2s)
query_xs = np.vstack((X1.flatten(), X2.flatten())).T


def prod_encode(x1,x2,enc1,enc2):
    phi1 = enc1.encode(x1)
    phi2 = enc2.encode(x2)
    return phi1 * phi2

def sum_encode(x1,x2,enc1,enc2):
    phi1 = enc1.encode(x1)
    phi2 = enc2.encode(x2)
    return (phi1 + phi2) / 2

# x1_origin = x1_space.encode([[10]])
# x2_origin = x2_space.encode([[10]])

prod_origin = prod_encode([[0]],[[0]],x1_space, x2_space)
sum_origin = sum_encode([[0]],[[0]],x1_space, x2_space)


query_sum_phis = sum_encode(query_xs[:,0][:,None], query_xs[:,1][:,None], x1_space, x2_space)
query_prod_phis = prod_encode(query_xs[:,0][:,None], query_xs[:,1][:,None], x1_space, x2_space)

neg_sum_origin = sum_encode([[-10]],[[-10]], x1_space, x2_space)
neg_prod_origin = prod_encode([[-10]],[[-10]], x1_space, x2_space)

pos_sum_origin = sum_encode([[10]],[[10]], x1_space, x2_space)
pos_prod_origin = prod_encode([[10]],[[10]], x1_space, x2_space)

neg_sum_sims = query_sum_phis | (sum_origin * neg_sum_origin)
neg_prod_sims = query_prod_phis | (prod_origin * neg_prod_origin)

sum_sims = query_sum_phis | sum_origin
prod_sims = query_prod_phis | prod_origin

pos_sum_sims = query_sum_phis | (sum_origin * pos_sum_origin)
pos_prod_sims = query_prod_phis | (prod_origin * pos_prod_origin)



axs_bot = subfigs[1].subplots(2, 3, subplot_kw={"projection": "3d"})
axs_bot[0,0].plot_surface(X1, X2, neg_sum_sims.reshape((100,100)), cmap=c_map)
axs_bot[0,0].set_title(r'$\phi(\mathbf{x}_{0}) \circledast \phi(-\Delta\mathbf{x})$', fontsize=font_size)

axs_bot[0,1].plot_surface(X1, X2, sum_sims.reshape((100,100)), cmap=c_map)
axs_bot[0,1].set_title(r'$\phi(\mathbf{x}_{0})$', fontsize=font_size)

axs_bot[0,2].plot_surface(X1, X2, pos_sum_sims.reshape((100,100)), cmap=c_map)
axs_bot[0,2].set_title(r'$\phi(\mathbf{x}_{0}) \circledast \phi(\Delta\mathbf{x})$', fontsize=font_size)



axs_bot[1,0].plot_surface(X1, X2, neg_prod_sims.reshape((100,100)), cmap=c_map)
# ax[1,0].set_title(r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$')

axs_bot[1,1].plot_surface(X1, X2, prod_sims.reshape((100,100)), cmap=c_map)
# ax[1,1].set_title(r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$')

axs_bot[1,2].plot_surface(X1, X2, pos_prod_sims.reshape((100,100)), cmap=c_map)
# ax[1,2].set_title()

z_label = [r'$\mathrm{sinc}(x_{1}) + \mathrm{sinc}(x_{2})$',r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$']

for row in range(2):
    for col in range(3):
        axs_bot[row,col].set(xticklabels=[],
                   yticklabels=[],
                   zticklabels=[])
        axs_bot[row,col].set_xlabel(r'$x_{1}$', fontsize=font_size)
        axs_bot[row,col].set_ylabel(r'$x_{2}$', fontsize=font_size)
        
        tmp_planes = axs_bot[row,col].zaxis._PLANES 
        axs_bot[row,col].zaxis._PLANES = ( tmp_planes[2], tmp_planes[3], 
                     tmp_planes[0], tmp_planes[1], 
                     tmp_planes[4], tmp_planes[5])
        # rotate label
        axs_bot[row,col].zaxis.set_rotate_label(False)  # disable automatic rotation
        
        if col == 0:
            axs_bot[row,col].set_zlabel(z_label[row], rotation=90, fontsize=font_size)
        axs_bot[row,col].grid(False)
        axs_bot[row,col].xaxis.labelpad = -10
        axs_bot[row,col].yaxis.labelpad = -10
        axs_bot[row,col].zaxis.labelpad = -10
    ### end for
### end for

# kernels_2d = plt.gcf()
# figure_utils.save(kernels_2d, 'shifting-encoding.pdf')
subfigs[0].text(0.05, 0.92, '\\textbf{A}', size=12, va="baseline", ha="left")
subfigs[1].text(0.05, 0.92, '\\textbf{B}', size=12, va="baseline", ha="left")
figure_utils.save(compositional_fig, 'compositional-figure.pdf')
# plt.show()

In [4]:
a = np.random.normal(loc=0,scale=1, size=(1000,))
a /= np.linalg.norm(a)
a = a[:,None]

def linear_kernel_vec(x,x_p, a):
    vec_x = np.einsum('dn,mn->dm',a, x - 1)
    vec_x_p = np.einsum('dn,mn->dm',a, x_p - 1)
    return np.einsum('dn,dm->nm',vec_x, vec_x_p)


k_vals = linear_kernel_vec(np.array([[0]]), np.linspace(-2,2,100)[:,None], a)
# print(k_vals[0])
plt.figure()
plt.plot(np.linspace(-2,2,100), k_vals.flatten())

# ks = linear_kernel_vec(query_xs[:,0][:,None].T,query_xs[:,1][:,None].T, a)
# plt.imshow(ks.reshape((100,100)), origin='lower')
# plt.colorbar()

# fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
# ax.plot_surface(X1, X2, ks.reshape((100,100)), cmap=c_map)